# jaraco.functools Package Testing

This notebook tests the jaraco.functools package, which provides functional programming utilities including caching, composition, and method decorators.

## Example 1: Function Composition and Utilities

In [ ]:
import jaraco.functools
import time

# Test function composition and caching
print("Testing jaraco.functools utilities:")

@jaraco.functools.once
def expensive_computation():
    print("Performing expensive computation...")
    time.sleep(0.1)  # Simulate work
    return "Computed result"

# Test the once decorator
print("First call:")
result1 = expensive_computation()
print(f"Result: {result1}")

print("\nSecond call:")
result2 = expensive_computation()
print(f"Result: {result2}")

print(f"Results are same object: {result1 is result2}")

# Test method combination
def add_one(x):
    return x + 1

def multiply_two(x):
    return x * 2

def square(x):
    return x ** 2

# Compose functions
composed = jaraco.functools.compose(square, multiply_two, add_one)
result = composed(5)  # ((5 + 1) * 2) ** 2 = 144
print(f"\nComposed function result: {result}")

## Example 2: Caching Decorators

In [ ]:
class Calculator:
    def __init__(self):
        self.calculation_count = 0
    
    # Using jaraco.functools.method_cache with functools.lru_cache
    @jaraco.functools.method_cache
    def expensive_calculation(self, n):
        self.calculation_count += 1
        print(f"Performing expensive calculation for {n} - count: {self.calculation_count}")
        time.sleep(0.1)  # Simulate expensive operation
        return n ** 2 + n + 1

calc = Calculator()
print(f"calc.expensive_calculation(5) = {calc.expensive_calculation(5)}")
print(f"calc.expensive_calculation(5) again = {calc.expensive_calculation(5)}")  # Should be cached
print(f"calc.expensive_calculation(3) = {calc.expensive_calculation(3)}")

## Example 3: Method and Property Utilities

In [ ]:
import jaraco.functools
import time

# Test method utilities
class TestClass:
    def __init__(self):
        self.value = 0
        self._computed_expensive = None
    
    @jaraco.functools.method_cache
    def expensive_method(self, x):
        print(f"Computing expensive method with {x}")
        time.sleep(0.05)  # Simulate work
        return x ** 2 + self.value
    
    @property
    @jaraco.functools.once
    def expensive_property(self):
        print("Computing expensive property")
        time.sleep(0.05)  # Simulate work
        return "Expensive result"

# Test the class
obj = TestClass()
print("Testing method cache:")
print(f"First call: {obj.expensive_method(5)}")
print(f"Second call: {obj.expensive_method(5)}")
print(f"Different arg: {obj.expensive_method(6)}")

print("\nTesting once property:")
print(f"First access: {obj.expensive_property}")
print(f"Second access: {obj.expensive_property}")

# Test with different instance
obj2 = TestClass()
print(f"\nDifferent instance: {obj2.expensive_method(5)}")

## Example 4: Function Transformation

In [ ]:
import jaraco.functools
import functools

# Test function transformation utilities
def original_function(a, b, c=10):
    return a + b + c

print("Testing function transformations:")
print(f"Original: {original_function(1, 2)}")

# jaraco.functools.apply is a decorator that transforms results
# It applies a transformation function to the result of the decorated function
@jaraco.functools.apply(lambda x: x * 2)  # Double the result
def doubled_result_func(a, b, c=10):
    return a + b + c

print(f"With apply decorator (doubled): {doubled_result_func(1, 2)}")

# For partial application, use standard functools.partial
partial_func = functools.partial(original_function, 1)
print(f"After partial application: {partial_func(2)}")

# You can also use jaraco.functools.apply with other transformations
@jaraco.functools.apply(reversed)
def get_numbers(start):
    """Get a range of numbers"""
    return range(start, start + 3)

print(f"Reversed range: {list(get_numbers(4))}")

# save_method_args is for CLASS METHODS, not standalone functions
class ExampleClass:
    @jaraco.functools.save_method_args
    def documented_method(self, x, y, z=None):
        """A well-documented method."""
        return f"Args: {x}, {y}, {z}"
    
    def show_saved_args(self):
        """Show the saved arguments from the last method call"""
        if hasattr(self, '_saved_documented_method'):
            return self._saved_documented_method
        return "No saved arguments found"

# Test the corrected method usage
example = ExampleClass()
print(f"\nMethod result: {example.documented_method('a', 'b', 'c')}")
print(f"Saved args: {example.show_saved_args()}")

# For standalone functions, you can use other jaraco.functools utilities
@jaraco.functools.apply(str.upper)
def get_greeting(name):
    """Get a greeting message"""
    return f"hello {name}"

print(f"Uppercase greeting: {get_greeting('world')}")

# Example of method_cache (similar to functools.lru_cache but for methods)
class Calculator:
    @jaraco.functools.method_cache
    def expensive_calculation(self, n):
        """Simulate an expensive calculation"""
        print(f"Computing for {n}...")
        return n ** 2 + n

calc = Calculator()
print(f"\nFirst call: {calc.expensive_calculation(5)}")
print(f"Second call (cached): {calc.expensive_calculation(5)}")
print(f"Different input: {calc.expensive_calculation(3)}")

# Test retry functionality - CORRECTED parameter usage
try:
    attempt_count = 0
    
    # The retry decorator uses 'tries' not 'times', and other parameters
    @jaraco.functools.retry(tries=3, delay=0.1)
    def unreliable_function():
        global attempt_count
        attempt_count += 1
        print(f"Attempt {attempt_count}")
        if attempt_count < 3:
            raise ConnectionError("Simulated failure")
        return "Success after retries"
    
    result = unreliable_function()
    print(f"\nRetry result: {result}")
    
except (AttributeError, TypeError) as e:
    print(f"\nRetry functionality issue: {e}")
    print("Let's try a simpler approach or check available parameters")
    
    def simple_retry(func, max_attempts=3):
        """Simple retry wrapper"""
        for attempt in range(max_attempts):
            try:
                return func()
            except Exception as e:
                if attempt == max_attempts - 1:
                    raise
                print(f"Attempt {attempt + 1} failed: {e}")
        
    def unreliable_function_v2():
        global attempt_count
        attempt_count += 1
        print(f"Attempt {attempt_count}")
        if attempt_count < 6:  # Reset counter for this test
            raise ConnectionError("Simulated failure")
        return "Success with custom retry"
    
    try:
        attempt_count = 3  # Reset counter
        result = simple_retry(unreliable_function_v2, 3)
        print(f"Custom retry result: {result}")
    except Exception as e:
        print(f"All attempts failed: {e}")
        raise

# Additional jaraco.functools utilities demonstration
print("\n" + "="*50)
print("Additional jaraco.functools utilities:")

# once decorator - ensures a function only runs once
@jaraco.functools.once
def initialize_system():
    print("System initialized!")
    return "initialization_complete"

print("First call:", initialize_system())
print("Second call:", initialize_system())  # Won't print "System initialized!" again
print("Third call:", initialize_system())

# pass_none decorator - allows None values to pass through without processing
@jaraco.functools.pass_none
def process_value(value):
    return value.upper()

print(f"\nWith string: {process_value('hello')}")
print(f"With None: {process_value(None)}")  # Returns None without error

## Example 5: Advanced Function Utilities

In [ ]:
import jaraco.functools
import inspect

# Test advanced function utilities
def sample_function(a, b, *args, **kwargs):
    """Sample function for testing."""
    return f"a={a}, b={b}, args={args}, kwargs={kwargs}"

print("Testing advanced utilities:")

# Test function signature preservation
@jaraco.functools.once
def wrapped_function(x, y=5):
    return x * y

print(f"Wrapped function signature: {inspect.signature(wrapped_function)}")
print(f"Result: {wrapped_function(3)}")
print(f"Second call: {wrapped_function(3)}")

# Test pass_none decorator if available
try:
    @jaraco.functools.pass_none
    def handle_none(value):
        return value.upper() if value else "DEFAULT"
    
    print(f"\npass_none with value: {handle_none('hello')}")
    print(f"pass_none with None: {handle_none(None)}")
    
except AttributeError:
    print("\npass_none not available in this version")

# Test apply decorator variations
@jaraco.functools.apply(str.upper)
def get_message():
    return "hello world"

result = get_message()
print(f"\nApply decorator result: {result}")

# Test function metadata preservation
original_name = sample_function.__name__
original_doc = sample_function.__doc__

wrapped_sample = jaraco.functools.once(sample_function)
print(f"\nOriginal name: {original_name}")
print(f"Wrapped name: {wrapped_sample.__name__}")
print(f"Name preserved: {original_name == wrapped_sample.__name__}")